In [19]:

import os
import sys
from pathlib import Path

import yaml
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

# Locate the project root
project_root = Path.cwd().resolve().parent

# Locate the RAG service
rag_service_root = project_root / "backend" / "rag-service"

# Make the RAG service importable
sys.path.insert(0, str(rag_service_root))

# Import RAG components
##from src.rag.embedding_models.instance import embedding_manager
##from src.rag.vector_store.instance import vector_store
from src.rag.embedding_models.embedding_factory import EmbeddingFactory
from src.rag.chunking_strategy.contextual_chunking.contextual_chunking import Contextualizer
from src.rag.llms.llama import LlamaManager
from src.rag.vector_store.chroma_store import ChromaVectorStore


print("Project root:", project_root)
print("RAG service root:", rag_service_root)
print("RAG service exists:", rag_service_root.exists())


Project root: C:\Users\gabri\Documents\capstone-projects\airline-support-bot
RAG service root: C:\Users\gabri\Documents\capstone-projects\airline-support-bot\backend\rag-service
RAG service exists: True


In [2]:

# Load Markdown documents from the knowledge base

RAW_DATA_DIR = rag_service_root / "data" / "raw"

loader = DirectoryLoader(
    str(RAW_DATA_DIR),
    glob="**/*.md",
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Knowledge base directory: {RAW_DATA_DIR}")
print(f"Number of documents: {len(documents)}")


Knowledge base directory: C:\Users\gabri\Documents\capstone-projects\airline-support-bot\backend\rag-service\data\raw
Number of documents: 30


In [3]:
## adding extra metadata to the documents
for document in documents:
    content = document.page_content

    # Split the front matter from the Markdown content
    if content.startswith("---"):
        _, front_matter, markdown = content.split("---", 2)

        # Convert YAML front matter into a Python dictionary
        metadata = yaml.safe_load(front_matter)

        # Add your metadata to LangChain's existing metadata
        document.metadata.update(metadata)

        # Remove the metadata from the actual page content
        document.page_content = markdown

In [4]:
## creating chunks from the documents
## 1. Creating the text splitter
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "section"),
        ("##", "subsection"),
        ("###", "subsubsection")
    ]
)

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [5]:
chunks = []

for document in documents:

    # First split according to Markdown headings
    sections = markdown_splitter.split_text(
        document.page_content
    )

    for section in sections:

        # Carry the original document metadata into the section
        section.metadata.update(document.metadata)

        # Split the section further if it is too large
        smaller_chunks = recursive_splitter.split_documents(
            [section]
        )

        for chunk in smaller_chunks:

            # Give every final chunk a unique ID
            chunk.metadata["chunk_id"] = (
                f"{document.metadata.get('document_id', 'UNKNOWN')}"
                f"-chunk-{len(chunks) + 1:03d}"
            )

            chunks.append(chunk)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 279


In [20]:
embedding_manager =EmbeddingFactory.create_embedding_model("gemini")  
#embeddings =  embedding_manager.create_embeddings(chunks)


In [8]:
vector_store.add_chunks(
   chunks,
   embeddings
)

In [3]:
import chromadb

database_paths = {
    "Project root": project_root / "data" / "processed" / "vector_store",
    "RAG service": rag_service_root / "data" / "processed" / "vector_store"
}

for database_name, path in database_paths.items():

    print(f"\n{'=' * 45}")
    print(f"DATABASE: {database_name}")
    print(f"PATH: {path}")

    client = chromadb.PersistentClient(path=str(path))

    collections = client.list_collections()

    if not collections:
        print("No collections found.")

    for collection in collections:
        print(
            f"Collection: {collection.name} | "
            f"Chunks: {collection.count()}"
        )


DATABASE: Project root
PATH: C:\Users\gabri\Documents\capstone-projects\airline-support-bot\data\processed\vector_store
Collection: Markdownrecursive_gemini | Chunks: 279
Collection: Markdownrecursive_allmini | Chunks: 279
Collection: kenya_airways_gemini | Chunks: 0

DATABASE: RAG service
PATH: C:\Users\gabri\Documents\capstone-projects\airline-support-bot\backend\rag-service\data\processed\vector_store
Collection: kenya_airways_gemini | Chunks: 279


In [9]:
import numpy as np
from data.raw.questions.question_dataset import QUESTION_DATASET
def load_or_create_query_embeddings(
    embedding_manager,
    question_dataset,
    cache_path
):
    # Extract question texts
    question_texts = [
        question["question_text"]
        for question in question_dataset
    ]

    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    # Load cached embeddings if available
    if cache_path.exists():
        query_embeddings = np.load(cache_path)
        print(f"Loaded cached embeddings: {cache_path.name}")

    else:
        # Generate embeddings
        query_embeddings = embedding_manager.create_query_embedding(
            question_texts
        )

        query_embeddings = np.asarray(
            query_embeddings,
            dtype=np.float32
        )

        # Save embeddings for future experiments
        np.save(cache_path, query_embeddings)

        print(f"Generated and saved embeddings: {cache_path.name}")

    print("Number of questions:", len(question_texts))
    print("Embedding shape:", query_embeddings.shape)

    return query_embeddings

In [8]:
def evaluate_relevance(retrieved_chunks, relevant_chunks):
    """
    Determine relevance by matching retrieved chunk IDs
    against the expected relevant chunk IDs.
    """

    relevant_chunk_ids = {
        chunk["chunk_id"] for chunk in relevant_chunks
    }

    relevance_labels = []

    for chunk in retrieved_chunks:
        is_relevant = chunk["chunk_id"] in relevant_chunk_ids
        relevance_labels.append(is_relevant)

    return relevance_labels

In [7]:
from src.rag.evaluation_metrics.evaluation import (
    calculate_precision,
    calculate_hit_at_k,
    calculate_mrr
)

def evaluate_retrieval(
    question_dataset,
    query_embeddings,
    vector_store,
    top_k=5
):
    evaluation_results = []

    for i, question in enumerate(question_dataset):

        query = question["question_text"]
        query_embedding = query_embeddings[i]

        # Retrieve candidate chunks
        retrieved = vector_store.retrieve(
            query_embedding.tolist(),
            top_k=top_k
        )

        retrieved_chunks = []

        for rank in range(len(retrieved["ids"][0])):

            chunk = {
                "rank": rank + 1,
                "chunk_id": retrieved["ids"][0][rank],
                "section": retrieved["metadatas"][0][rank].get("section"),
                "distance": retrieved["distances"][0][rank],
                "content": retrieved["documents"][0][rank]
            }

            retrieved_chunks.append(chunk)

        # Determine relevance using expected chunk IDs
        relevance_labels = evaluate_relevance(
            retrieved_chunks,
            question["chunks"]
        )

        # Calculate metrics
        precision = calculate_precision(relevance_labels)
        hit = calculate_hit_at_k(relevance_labels)
        reciprocal_rank = calculate_mrr(relevance_labels)

        evaluation_results.append({
            "question_id": question["question_id"],
            "question": query,
            "relevant_chunks": question["chunks"],
            "retrieved_chunks": retrieved_chunks,
            "relevance_labels": relevance_labels,
            "precision_at_5": precision,
            "hit_at_5": hit,
            "reciprocal_rank": reciprocal_rank
        })

        if (i + 1) % 50 == 0:
            print(
                f"Processed {i + 1}/{len(question_dataset)} questions"
            )

    print(
        f"\nEvaluation completed for {len(evaluation_results)} questions."
    )

    return evaluation_results

In [10]:
def inspect_retrieval_result(evaluation_results, index=0):

    sample = evaluation_results[index]

    print("Question ID:", sample["question_id"])
    print("Question:", sample["question"])

    print("\nExpected relevant chunks:")
    for chunk in sample["relevant_chunks"]:
        print(f"- {chunk['chunk_id']} | {chunk['section']}")

    print("\nRetrieved chunks:")
    for chunk, is_relevant in zip(
        sample["retrieved_chunks"],
        sample["relevance_labels"]
    ):
        print(f"\nRank: {chunk['rank']}")
        print(f"Chunk ID: {chunk['chunk_id']}")
        print(f"Section: {chunk['section']}")
        print(f"Distance: {chunk['distance']:.4f}")
        print(f"Relevant: {is_relevant}")
        print(f"Content: {chunk['content'][:300]}...")

    print("\nQuestion Metrics:")
    print(f"Precision@5: {sample['precision_at_5']:.4f}")
    print(f"Hit@5: {sample['hit_at_5']}")
    print(f"Reciprocal Rank: {sample['reciprocal_rank']:.4f}")

In [11]:
def inspect_multiple_retrieval_results(evaluation_results, count=5):

    for result in evaluation_results[:count]:

        print("\n" + "=" * 70)
        print("Question ID:", result["question_id"])
        print("Question:", result["question"])

        print("\nExpected chunks:")
        for chunk in result["relevant_chunks"]:
            print("-", chunk["chunk_id"])

        print("\nRetrieved ranking:")
        for chunk, relevant in zip(
            result["retrieved_chunks"],
            result["relevance_labels"]
        ):
            print(
                f"Rank {chunk['rank']} | "
                f"{chunk['chunk_id']} | "
                f"Relevant: {relevant}"
            )

In [12]:
import numpy as np

def calculate_retrieval_metrics(evaluation_results):

    precision_at_5 = np.mean([
        result["precision_at_5"]
        for result in evaluation_results
    ])

    hit_at_5 = np.mean([
        result["hit_at_5"]
        for result in evaluation_results
    ])

    mrr = np.mean([
        result["reciprocal_rank"]
        for result in evaluation_results
    ])

    metrics = {
        "Precision@5": precision_at_5,
        "Hit@5": hit_at_5,
        "MRR": mrr
    }

    print("RETRIEVAL PERFORMANCE")
    print("=" * 40)
    print(f"Total questions: {len(evaluation_results)}")
    print(f"Precision@5:    {precision_at_5:.4f}")
    print(f"Hit@5:          {hit_at_5:.4f}")
    print(f"MRR:            {mrr:.4f}")

    return metrics

In [13]:
import pandas as pd

def compare_retrieval_metrics(
    baseline_metrics,
    new_metrics,
    baseline_name="Baseline",
    new_name="After Reranking"
):
    comparison = pd.DataFrame({
        "Metric": ["Precision@5", "Hit@5", "MRR"],
        baseline_name: [
            baseline_metrics["Precision@5"],
            baseline_metrics["Hit@5"],
            baseline_metrics["MRR"]
        ],
        new_name: [
            new_metrics["Precision@5"],
            new_metrics["Hit@5"],
            new_metrics["MRR"]
        ]
    })

    comparison["Absolute Change"] = (
        comparison[new_name] - comparison[baseline_name]
    )

    comparison["Relative Improvement (%)"] = (
        comparison["Absolute Change"] /
        comparison[baseline_name]
    ) * 100

    comparison = comparison.round(4)

    print(f"RETRIEVAL PERFORMANCE: {baseline_name} VS {new_name}")
    print("=" * 65)

    display(comparison)

    return comparison

In [14]:
import copy

from src.rag.rerankers.instance import reranker

def evaluate_reranking(
    evaluation_results,
    reranker,
    top_k=5
):
    reranked_results = []

    for i, result in enumerate(evaluation_results):

        query = result["question"]

        # Preserve the original retrieved candidates
        candidates = copy.deepcopy(result["retrieved_chunks"])

        # Rerank the candidates
        reranked_chunks = reranker.rerank(
            query,
            candidates,
            top_k=top_k
        )

        # Evaluate against the existing ground truth
        relevance_labels = evaluate_relevance(
            reranked_chunks,
            result["relevant_chunks"]
        )

        # Calculate metrics
        reranked_results.append({
            "question_id": result["question_id"],
            "question": query,
            "relevant_chunks": result["relevant_chunks"],
            "retrieved_chunks": reranked_chunks,
            "relevance_labels": relevance_labels,
            "precision_at_5": calculate_precision(relevance_labels),
            "hit_at_5": calculate_hit_at_k(relevance_labels),
            "reciprocal_rank": calculate_mrr(relevance_labels)
        })

        if (i + 1) % 50 == 0:
            print(
                f"Processed {i + 1}/{len(evaluation_results)} questions"
            )

    print(
        f"\nReranking completed: "
        f"{len(reranked_results)}/{len(evaluation_results)} questions"
    )

    return reranked_results

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

# Experiment 1A: Baseline Retrieval Evaluation Using Markdown and Recursive Character Splitting with MiniLM Embeddings

## 1. Objective

This experiment establishes a baseline for evaluating the retrieval performance of the Kenya Airways customer and staff support chatbot. It investigates how effectively the system retrieves relevant information when documents are divided using Markdown header splitting followed by recursive character splitting, with `all-MiniLM-L6-v2` used to generate document and query embeddings.

## 2. Experimental Configuration

The original Markdown knowledge-base documents are processed using the following chunking strategy:

1. **Markdown header splitting:** Documents are initially divided according to their heading structure to preserve their logical organisation.
2. **Recursive character splitting:** The resulting sections are further divided into smaller chunks using a chunk size of 1,000 characters and an overlap of 200 characters.
3. **Embedding generation:** The `all-MiniLM-L6-v2` sentence-transformer model generates vector representations of the resulting document chunks and evaluation queries.
4. **Vector storage and retrieval:** The document embeddings are stored in ChromaDB and used to retrieve the five most relevant chunks for each evaluation query.

## 3. Evaluation

The retrieval system is evaluated using the same set of 609 evaluation questions and their associated relevance judgements. Performance is assessed using:

- **Hit Rate@5:** Measures the proportion of queries for which at least one relevant document appears among the five retrieved results.
- **Mean Reciprocal Rank (MRR@5):** Measures the ranking position of the first relevant result.
- **Mean Average Precision (MAP@5):** Measures the overall ranking quality of relevant results within the top five retrieved results.

## 4. Purpose of the Baseline

The results from this experiment serve as a baseline for comparison with subsequent experiments. In particular, the same Markdown and recursive character splitting strategy will be evaluated using Gemini embeddings to investigate the effect of changing the embedding model while keeping the chunking strategy and evaluation conditions consistent.

Generated and saved MiniLM question embeddings.
Number of questions: 609
Embedding shape: (609, 384)


In [33]:
vector_store = ChromaVectorStore(collection_name="Markdownrecursive_allmini")
embedding_manager = EmbeddingFactory.create_embedding_model("all-MiniLM-L6-v2")
print("Collection:", vector_store.collection.name)
print("Stored chunks:", vector_store.collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Collection: Markdownrecursive_allmini
Stored chunks: 279


In [34]:
minilm_embeddings_path = (
    project_root / "evaluation_artifacts" /
    "embeddings" / "minilm_question_embeddings.npy"
)

minilm_query_embeddings = load_or_create_query_embeddings(
    embedding_manager=embedding_manager,
    question_dataset=QUESTION_DATASET,
    cache_path=minilm_embeddings_path
)

Loaded cached embeddings: minilm_question_embeddings.npy
Number of questions: 609
Embedding shape: (609, 384)


In [35]:
minilm_results = evaluate_retrieval(
    question_dataset=QUESTION_DATASET,
    query_embeddings=minilm_query_embeddings,
    vector_store=vector_store,
    top_k=5
)

Processed 50/609 questions
Processed 100/609 questions
Processed 150/609 questions
Processed 200/609 questions
Processed 250/609 questions
Processed 300/609 questions
Processed 350/609 questions
Processed 400/609 questions
Processed 450/609 questions
Processed 500/609 questions
Processed 550/609 questions
Processed 600/609 questions

Evaluation completed for 609 questions.


In [36]:
inspect_retrieval_result(minilm_results, index=0)

Question ID: Q001
Question: What are the steps for booking a Kenya Airways flight online?

Expected relevant chunks:
- KQ-BOOK-001-chunk-071 | Book a Flight - How to Book a Flight

Retrieved chunks:

Rank: 1
Chunk ID: KQ-DISR-001-chunk-160
Section: Frequently Asked Questions (FAQs)
Distance: 0.3943
Relevant: False
Content: Visit the Kenya Airways website and select **Manage Booking**.  
---...

Rank: 2
Chunk ID: KQ-BOOK-001-chunk-070
Section: Book a Flight
Distance: 0.4775
Relevant: False
Content: Passengers can search for and book Kenya Airways flights online to destinations served by the airline. The online booking system allows passengers to select their travel preferences and compare available fares before completing their booking.  
---...

Rank: 3
Chunk ID: KQ-DISR-001-chunk-146
Section: How to Rebook
Distance: 0.5154
Relevant: False
Content: 1. Visit the Kenya Airways Self-Rebooking portal.
2. Enter your booking reference or electronic ticket number together with your last name.

In [37]:
inspect_multiple_retrieval_results(minilm_results, count=5)


Question ID: Q001
Question: What are the steps for booking a Kenya Airways flight online?

Expected chunks:
- KQ-BOOK-001-chunk-071

Retrieved ranking:
Rank 1 | KQ-DISR-001-chunk-160 | Relevant: False
Rank 2 | KQ-BOOK-001-chunk-070 | Relevant: False
Rank 3 | KQ-DISR-001-chunk-146 | Relevant: False
Rank 4 | KQ-BOOK-003-chunk-088 | Relevant: False
Rank 5 | KQ-BOOK-001-chunk-071 | Relevant: True

Question ID: Q002
Question: What types of trips can I book online?

Expected chunks:
- KQ-BOOK-001-chunk-072

Retrieved ranking:
Rank 1 | KQ-BOOK-001-chunk-071 | Relevant: False
Rank 2 | KQ-DISR-001-chunk-145 | Relevant: False
Rank 3 | KQ-BOOK-001-chunk-070 | Relevant: False
Rank 4 | KQ-BOOK-006-chunk-065 | Relevant: False
Rank 5 | KQ-DISR-001-chunk-156 | Relevant: False

Question ID: Q003
Question: Can I book a one-way or multi-city flight?

Expected chunks:
- KQ-BOOK-001-chunk-072

Retrieved ranking:
Rank 1 | KQ-BOOK-001-chunk-071 | Relevant: False
Rank 2 | KQ-BOOK-001-chunk-072 | Relevant: Tr

In [39]:
minilm_metrics = calculate_retrieval_metrics(minilm_results)

RETRIEVAL PERFORMANCE
Total questions: 609
Precision@5:    0.1888
Hit@5:          0.8588
MRR:            0.6880


In [40]:
minilm_reranked_results = evaluate_reranking(
    evaluation_results=minilm_results,
    reranker=reranker,
    top_k=5
)

Processed 50/609 questions
Processed 100/609 questions
Processed 150/609 questions
Processed 200/609 questions
Processed 250/609 questions
Processed 300/609 questions
Processed 350/609 questions
Processed 400/609 questions
Processed 450/609 questions
Processed 500/609 questions
Processed 550/609 questions
Processed 600/609 questions

Reranking completed: 609/609 questions


In [45]:
minilm_reranked_metrics = calculate_retrieval_metrics(minilm_results)

RETRIEVAL PERFORMANCE
Total questions: 609
Precision@5:    0.1888
Hit@5:          0.8588
MRR:            0.6880


In [46]:
minilm_comparison = compare_retrieval_metrics(
    baseline_metrics=minilm_metrics,
    new_metrics=minilm_reranked_metrics,
    baseline_name="MiniLM Baseline",
    new_name="MiniLM + Reranking"
)

RETRIEVAL PERFORMANCE: MiniLM Baseline VS MiniLM + Reranking


,Metric,MiniLM Baseline,MiniLM + Reranking,Absolute Change,Relative Improvement (%)
0,Precision@5,0.1888,0.1888,0.0,0.0
1,Hit@5,0.8588,0.8588,0.0,0.0
2,MRR,0.6880,0.6880,0.0,0.0


# Experiment 2: Gemini Embeddings

## Objective
The objective of this experiment is to evaluate the retrieval performance of the Gemini embedding model using the same evaluation dataset and ground-truth chunk identifiers used in Experiment 1.

This experiment aims to determine whether Gemini embeddings improve the retrieval of relevant knowledge-base chunks compared to the MiniLM embedding model.

## Experimental Configuration

| Parameter | Configuration |
|---|---|
| Embedding model | Gemini |
| Chunking strategy | Markdown Header + Recursive Character Splitting |
| Number of chunks | 279 |
| Evaluation questions | 609 |
| Ground truth | Manually identified relevant chunk IDs |
| Retrieval method | Vector similarity search |
| Top-K | 5 |
| Evaluation metrics | Precision@5, Hit@5, MRR |
| Reranking model | Cross-Encoder MiniLM |

## Experiment Procedure

The experiment is conducted in two stages:

### Stage 1: Baseline Retrieval
- Generate query embeddings using the Gemini embedding model.
- Retrieve the five most similar chunks for each evaluation question.
- Compare retrieved chunk IDs against the manually established ground truth.
- Calculate Precision@5, Hit@5 and Mean Reciprocal Rank (MRR).

### Stage 2: Cross-Encoder Reranking
- Pass the five retrieved candidate chunks and their corresponding questions to the cross-encoder.
- Calculate relevance scores for each question-chunk pair.
- Reorder the retrieved chunks according to their cross-encoder scores.
- Recalculate Precision@5, Hit@5 and MRR.

## Results and Comparison

The retrieval results will be compared before and after reranking to assess the effect of the cross-encoder on retrieval performance.

| Metric | Gemini Baseline | Gemini + Reranking |
|---|---:|---:|
| Precision@5 | TBD | TBD |
| Hit@5 | TBD | TBD |
| MRR | TBD | TBD |

## Discussion

The results will be analysed to determine whether Gemini embeddings improve initial candidate retrieval and whether cross-encoder reranking further improves the ranking of relevant chunks.

In [47]:
vector_store = ChromaVectorStore(collection_name="Markdownrecursive_gemini")
embedding_manager = EmbeddingFactory.create_embedding_model("gemini")
print("Collection:", vector_store.collection.name)
print("Stored chunks:", vector_store.collection.count())

Collection: Markdownrecursive_gemini
Stored chunks: 279


In [48]:
## gemini question vectors
gemini_embeddings_path = (
    project_root
    / "evaluation_artifacts"
    / "embeddings"
    / "gemini_question_embeddings.npy"
)

gemini_query_embeddings = load_or_create_query_embeddings(
    embedding_manager=embedding_manager,
    question_dataset=QUESTION_DATASET,
    cache_path=gemini_embeddings_path
)

Loaded cached embeddings: gemini_question_embeddings.npy
Number of questions: 609
Embedding shape: (609, 3072)


In [49]:
gemini_results = evaluate_retrieval(
    question_dataset=QUESTION_DATASET,
    query_embeddings=gemini_query_embeddings,
    vector_store=vector_store,
    top_k=5
)

Processed 50/609 questions
Processed 100/609 questions
Processed 150/609 questions
Processed 200/609 questions
Processed 250/609 questions
Processed 300/609 questions
Processed 350/609 questions
Processed 400/609 questions
Processed 450/609 questions
Processed 500/609 questions
Processed 550/609 questions
Processed 600/609 questions

Evaluation completed for 609 questions.


In [50]:
inspect_retrieval_result(gemini_results, index=0)

inspect_multiple_retrieval_results(
    gemini_results,
    count=5
)

Question ID: Q001
Question: What are the steps for booking a Kenya Airways flight online?

Expected relevant chunks:
- KQ-BOOK-001-chunk-071 | Book a Flight - How to Book a Flight

Retrieved chunks:

Rank: 1
Chunk ID: KQ-BOOK-001-chunk-070
Section: Book a Flight
Distance: 0.2617
Relevant: False
Content: Passengers can search for and book Kenya Airways flights online to destinations served by the airline. The online booking system allows passengers to select their travel preferences and compare available fares before completing their booking.  
---...

Rank: 2
Chunk ID: KQ-DISR-001-chunk-160
Section: Frequently Asked Questions (FAQs)
Distance: 0.3750
Relevant: False
Content: Visit the Kenya Airways website and select **Manage Booking**.  
---...

Rank: 3
Chunk ID: KQ-DISR-001-chunk-146
Section: How to Rebook
Distance: 0.4014
Relevant: False
Content: 1. Visit the Kenya Airways Self-Rebooking portal.
2. Enter your booking reference or electronic ticket number together with your last name.

In [51]:
gemini_metrics = calculate_retrieval_metrics(gemini_results)

RETRIEVAL PERFORMANCE
Total questions: 609
Precision@5:    0.2128
Hit@5:          0.9475
MRR:            0.8242


In [52]:
gemini_reranked_results = evaluate_reranking(
    evaluation_results=gemini_results,
    reranker=reranker,
    top_k=5
)

Processed 50/609 questions
Processed 100/609 questions
Processed 150/609 questions
Processed 200/609 questions
Processed 250/609 questions
Processed 300/609 questions
Processed 350/609 questions
Processed 400/609 questions
Processed 450/609 questions
Processed 500/609 questions
Processed 550/609 questions
Processed 600/609 questions

Reranking completed: 609/609 questions


In [53]:
gemini_reranked_metrics = calculate_retrieval_metrics(
    gemini_reranked_results
)

RETRIEVAL PERFORMANCE
Total questions: 609
Precision@5:    0.2128
Hit@5:          0.9475
MRR:            0.8272


In [54]:
gemini_comparison = compare_retrieval_metrics(
    baseline_metrics=gemini_metrics,
    new_metrics=gemini_reranked_metrics,
    baseline_name="Gemini Baseline",
    new_name="Gemini + Reranking"
)

RETRIEVAL PERFORMANCE: Gemini Baseline VS Gemini + Reranking


,Metric,Gemini Baseline,Gemini + Reranking,Absolute Change,Relative Improvement (%)
0,Precision@5,0.2128,0.2128,0.000,0.0000
1,Hit@5,0.9475,0.9475,0.000,0.0000
2,MRR,0.8242,0.8272,0.003,0.3652


## Experiment 3: Gemini Embeddings with Contextual Chunking

This experiment evaluates the retrieval performance of the system using Gemini embeddings and contextualized document chunks. The document collection uses Markdown header splitting followed by recursive character splitting, with contextual information added to each chunk to preserve its relationship with the broader document.

Unlike the previous experiments, which used the original document chunks, this experiment uses the contextualized chunks already stored in the application's ChromaDB vector store. The existing document embeddings are reused to avoid unnecessary re-embedding.

The evaluation uses the same dataset of 609 questions and their manually labelled relevant chunk IDs. Gemini query embeddings generated in the previous experiment are also reused to ensure consistency.

The retrieval process returns the top five candidate chunks for each question. Performance is evaluated using Precision@5, Hit@5 and Mean Reciprocal Rank (MRR).

The experiment aims to assess how contextual chunking, combined with Gemini embeddings, affects retrieval quality compared with the previous configurations.

In [55]:
rag_service_vector_store_path = (
    rag_service_root
    / "data"
    / "processed"
    / "vector_store"
)

contextual_gemini_vector_store = ChromaVectorStore(
    collection_name="kenya_airways_gemini",
    persist_directory=str(rag_service_vector_store_path)
)

In [56]:
print("Database:", rag_service_vector_store_path)
print("Collection:", contextual_gemini_vector_store.collection.name)
print("Chunks:", contextual_gemini_vector_store.collection.count())

Database: C:\Users\gabri\Documents\capstone-projects\airline-support-bot\backend\rag-service\data\processed\vector_store
Collection: kenya_airways_gemini
Chunks: 279


In [57]:
contextual_gemini_results = evaluate_retrieval(
    question_dataset=QUESTION_DATASET,
    query_embeddings=gemini_query_embeddings,
    vector_store=contextual_gemini_vector_store,
    top_k=5
)

Processed 50/609 questions
Processed 100/609 questions
Processed 150/609 questions
Processed 200/609 questions
Processed 250/609 questions
Processed 300/609 questions
Processed 350/609 questions
Processed 400/609 questions
Processed 450/609 questions
Processed 500/609 questions
Processed 550/609 questions
Processed 600/609 questions

Evaluation completed for 609 questions.


In [58]:
contextual_gemini_metrics = calculate_retrieval_metrics(
    contextual_gemini_results
)

RETRIEVAL PERFORMANCE
Total questions: 609
Precision@5:    0.2177
Hit@5:          0.9672
MRR:            0.8375


In [59]:
inspect_multiple_retrieval_results(
    contextual_gemini_results,
    count=5
)


Question ID: Q001
Question: What are the steps for booking a Kenya Airways flight online?

Expected chunks:
- KQ-BOOK-001-chunk-071

Retrieved ranking:
Rank 1 | KQ-BOOK-001-chunk-070 | Relevant: False
Rank 2 | KQ-BOOK-001-chunk-071 | Relevant: True
Rank 3 | KQ-BOOK-001-chunk-073 | Relevant: False
Rank 4 | KQ-DISR-001-chunk-160 | Relevant: False
Rank 5 | KQ-DISR-001-chunk-146 | Relevant: False

Question ID: Q002
Question: What types of trips can I book online?

Expected chunks:
- KQ-BOOK-001-chunk-072

Retrieved ranking:
Rank 1 | KQ-BOOK-001-chunk-072 | Relevant: True
Rank 2 | KQ-BOOK-001-chunk-071 | Relevant: False
Rank 3 | KQ-BOOK-001-chunk-070 | Relevant: False
Rank 4 | KQ-CHECK-002-chunk-124 | Relevant: False
Rank 5 | KQ-BOOK-003-chunk-088 | Relevant: False

Question ID: Q003
Question: Can I book a one-way or multi-city flight?

Expected chunks:
- KQ-BOOK-001-chunk-072

Retrieved ranking:
Rank 1 | KQ-BOOK-001-chunk-072 | Relevant: True
Rank 2 | KQ-BOOK-004-chunk-092 | Relevant: Fal

In [60]:
contextual_gemini_reranked_results = evaluate_reranking(
    evaluation_results=contextual_gemini_results,
    reranker=reranker,
    top_k=5
)

Processed 50/609 questions
Processed 100/609 questions
Processed 150/609 questions
Processed 200/609 questions
Processed 250/609 questions
Processed 300/609 questions
Processed 350/609 questions
Processed 400/609 questions
Processed 450/609 questions
Processed 500/609 questions
Processed 550/609 questions
Processed 600/609 questions

Reranking completed: 609/609 questions


In [64]:
contextual_gemini_reranked_metrics = calculate_retrieval_metrics(
    contextual_gemini_reranked_results
)

RETRIEVAL PERFORMANCE
Total questions: 609
Precision@5:    0.2177
Hit@5:          0.9672
MRR:            0.8766


In [65]:
contextual_gemini_comparison = compare_retrieval_metrics(
    baseline_metrics=contextual_gemini_metrics,
    new_metrics=contextual_gemini_reranked_metrics,
    baseline_name="Contextual Gemini",
    new_name="Contextual Gemini + Reranking"
)

RETRIEVAL PERFORMANCE: Contextual Gemini VS Contextual Gemini + Reranking


,Metric,Contextual Gemini,Contextual Gemini + Reranking,Absolute Change,Relative Improvement (%)
0,Precision@5,0.2177,0.2177,0.0000,0.000
1,Hit@5,0.9672,0.9672,0.0000,0.000
2,MRR,0.8375,0.8766,0.0391,4.673


In [66]:
qualitative_evaluation = []

for question in QUESTION_DATASET[:20]:
    qualitative_evaluation.append({
        "question_id": question["question_id"],
        "question": question["question_text"],

        # Ground truth
        "ground_truth_chunks": question["chunks"],
        "ground_truth_answer": None,

        # Generated responses
        "gemini_baseline": None,
        "gemini_reranked": None,
        "contextual_gemini": None,

        # Retrieved context
        "baseline_context": None,
        "reranked_context": None,
        "contextual_context": None
    })

print("Questions selected:", len(qualitative_evaluation))

Questions selected: 20


In [67]:


qualitative_df = pd.DataFrame([
    {
        "Question ID": item["question_id"],
        "Question": item["question"],
        "Ground Truth": item["ground_truth_answer"],
        "Gemini Baseline": item["gemini_baseline"],
        "Gemini + Reranking": item["gemini_reranked"],
        "Contextual Gemini": item["contextual_gemini"]
    }
    for item in qualitative_evaluation
])

display(qualitative_df)

,Question ID,Question,Ground Truth,Gemini Baseline,Gemini + Reranking,Contextual Gemini
0,Q001,What are the steps for booking a Kenya Airways...,None,None,None,None
1,Q002,What types of trips can I book online?,None,None,None,None
2,Q003,Can I book a one-way or multi-city flight?,None,None,None,None
3,Q004,What information do I need to provide when sea...,None,None,None,None
4,Q005,How can I change my seat after making a booking?,None,None,None,None
5,Q006,What can I do through Manage Booking?,None,None,None,None
6,Q007,Can I modify an existing booking and purchase ...,None,None,None,None
7,Q008,What factors can affect the changes I make to ...,None,None,None,None
8,Q009,Why might the fares shown when booking a fligh...,None,None,None,None
9,Q010,What determines whether a flight is available?,None,None,None,None


In [72]:
def get_result_by_question_id(results, question_id):
    return next(
        result for result in results
        if result["question_id"] == question_id
    )


qualitative_evaluation = []

for question in QUESTION_DATASET[:20]:

    question_id = question["question_id"]

    # Retrieve the corresponding results from each configuration
    baseline = get_result_by_question_id(
        gemini_results, question_id
    )

    reranked = get_result_by_question_id(
        gemini_reranked_results, question_id
    )

    contextual = get_result_by_question_id(
        contextual_gemini_reranked_results, question_id
    )

    qualitative_evaluation.append({
        "question_id": question_id,
        "question": question["question_text"],

        # Ground truth
        "ground_truth_chunks": question["chunks"],
        "ground_truth_answer": None,

        # Retrieved contexts
        "baseline_context": baseline["retrieved_chunks"],
        "reranked_context": reranked["retrieved_chunks"],
        "contextual_context": contextual["retrieved_chunks"],

        # Generated answers (we will populate these next)
        "baseline_response": None,
        "reranked_response": None,
        "contextual_response": None
    })

print("Questions selected:", len(qualitative_evaluation))

Questions selected: 20


In [73]:
qualitative_df = pd.DataFrame([
    {
        "Question ID": item["question_id"],
        "Question": item["question"],
        "Ground Truth Answer": item["ground_truth_answer"],
        "Gemini Baseline": item["baseline_response"],
        "Gemini + Reranking": item["reranked_response"],
        "Contextual Gemini + Reranking": item["contextual_response"]
    }
    for item in qualitative_evaluation
])

display(qualitative_df)

,Question ID,Question,Ground Truth Answer,Gemini Baseline,Gemini + Reranking,Contextual Gemini + Reranking
0,Q001,What are the steps for booking a Kenya Airways...,None,None,None,None
1,Q002,What types of trips can I book online?,None,None,None,None
2,Q003,Can I book a one-way or multi-city flight?,None,None,None,None
3,Q004,What information do I need to provide when sea...,None,None,None,None
4,Q005,How can I change my seat after making a booking?,None,None,None,None
5,Q006,What can I do through Manage Booking?,None,None,None,None
6,Q007,Can I modify an existing booking and purchase ...,None,None,None,None
7,Q008,What factors can affect the changes I make to ...,None,None,None,None
8,Q009,Why might the fares shown when booking a fligh...,None,None,None,None
9,Q010,What determines whether a flight is available?,None,None,None,None


In [74]:
def get_result_by_question_id(results, question_id):
    return next(
        result for result in results
        if result["question_id"] == question_id
    )


qualitative_evaluation = []

for question in QUESTION_DATASET[:20]:

    question_id = question["question_id"]

    baseline = get_result_by_question_id(
        gemini_results, question_id
    )

    reranked = get_result_by_question_id(
        gemini_reranked_results, question_id
    )

    contextual = get_result_by_question_id(
        contextual_gemini_reranked_results, question_id
    )

    qualitative_evaluation.append({
        "question_id": question_id,
        "question": question["question_text"],
        "ground_truth_chunks": question["chunks"],

        "baseline_context": baseline["retrieved_chunks"],
        "reranked_context": reranked["retrieved_chunks"],
        "contextual_context": contextual["retrieved_chunks"],

        "ground_truth_answer": None,
        "baseline_response": None,
        "reranked_response": None,
        "contextual_response": None
    })

print(f"Prepared {len(qualitative_evaluation)} questions.")

Prepared 20 questions.


In [75]:
for item in qualitative_evaluation[:2]:
    print("\nQUESTION:", item["question"])

    print("\nBASELINE CONTEXT:")
    for chunk in item["baseline_context"]:
        print(chunk["chunk_id"], chunk["content"][:200])

    print("\nRERANKED CONTEXT:")
    for chunk in item["reranked_context"]:
        print(chunk["chunk_id"], chunk["content"][:200])

    print("\nCONTEXTUAL CONTEXT:")
    for chunk in item["contextual_context"]:
        print(chunk["chunk_id"], chunk["content"][:200])


QUESTION: What are the steps for booking a Kenya Airways flight online?

BASELINE CONTEXT:
KQ-BOOK-001-chunk-070 Passengers can search for and book Kenya Airways flights online to destinations served by the airline. The online booking system allows passengers to select their travel preferences and compare availa
KQ-DISR-001-chunk-160 Visit the Kenya Airways website and select **Manage Booking**.  
---
KQ-DISR-001-chunk-146 1. Visit the Kenya Airways Self-Rebooking portal.
2. Enter your booking reference or electronic ticket number together with your last name.
3. View the available alternative flights.
4. Select the pre
KQ-BOOK-003-chunk-083 Kenya Airways supports several payment methods for booking flights and purchasing additional services.
KQ-BOOK-001-chunk-073 Passengers can manage an existing booking through the Kenya Airways **Manage Booking** service.  
After logging in with their booking details, passengers can:  
- Customize their booking.
- Select or 

RERANKED CONTEXT:
KQ-D

In [ ]:

qualitative_comparison = []

for question in QUESTION_DATASET[:20]:

    question_id = question["question_id"]

    baseline = get_result_by_question_id(
        gemini_results, question_id
    )

    reranked = get_result_by_question_id(
        gemini_reranked_results, question_id
    )

    contextual = get_result_by_question_id(
        contextual_gemini_reranked_results, question_id
    )

    qualitative_comparison.append({
        "Question ID": question_id,
        "Question": question["question_text"],

        "Gemini Baseline (Rank 1)": 
            baseline["retrieved_chunks"][0]["content"],

        "Gemini + Reranking (Rank 1)": 
            reranked["retrieved_chunks"][0]["content"],

        "Contextual + Reranking (Rank 1)": 
            contextual["retrieved_chunks"][0]["content"]
    })

qualitative_df = pd.DataFrame(qualitative_comparison)

display(qualitative_df)

,Question ID,Question,Gemini Baseline (Rank 1),Gemini + Reranking (Rank 1),Contextual + Reranking (Rank 1)
0,Q001,What are the steps for booking a Kenya Airways...,Passengers can search for and book Kenya Airwa...,1. Visit the Kenya Airways Self-Rebooking port...,Context: The TARGET CHUNK provides step-by-ste...
1,Q002,What types of trips can I book online?,- One-way\n- Return\n- Multi-city \n---,To book a flight online: \n1. Enter the depar...,Context: The TARGET CHUNK provides options for...
2,Q003,Can I book a one-way or multi-city flight?,- One-way\n- Return\n- Multi-city \n---,To book a flight online: \n1. Enter the depar...,Context: The TARGET CHUNK provides options for...
3,Q004,What information do I need to provide when sea...,To book a flight online: \n1. Enter the depar...,To book a flight online: \n1. Enter the depar...,Context: The TARGET CHUNK provides instruction...
4,Q005,How can I change my seat after making a booking?,- Seat options are subject to availability.\n-...,Passengers can manage an existing booking thro...,Context: The TARGET CHUNK explains how passeng...
5,Q006,What can I do through Manage Booking?,Passengers can manage an existing booking thro...,Passengers can manage an existing booking thro...,Context: The TARGET CHUNK explains how passeng...
6,Q007,Can I modify an existing booking and purchase ...,Passengers may upgrade by paying the fare diff...,Tickets upgraded through: \n- Last-Minute Air...,Context: The TARGET CHUNK explains that passen...
7,Q008,What factors can affect the changes I make to ...,Yes. The following remain unchanged: \n- Dest...,1. Confirm the passenger's booking reference a...,Context: This TARGET CHUNK outlines the step-b...
8,Q009,Why might the fares shown when booking a fligh...,- Flight availability depends on the selected ...,- Flight availability depends on the selected ...,Context: The TARGET CHUNK explains the factors...
9,Q010,What determines whether a flight is available?,- Flight availability depends on the selected ...,- Flight availability depends on the selected ...,Context: The TARGET CHUNK explains the eligibi...
